### 1. Raw Database from IMDC

In [ ]:
"""
==========================
Monta a macrobase `forecast_database` a partir das bases do desafio IMDC 2026.

Estrutura esperada em:
  C:/Users/User/OneDrive/Área de Trabalho/myenv/pyenv/2026_06_imdc_challenge/databases/
    ├── dengue.csv
    ├── climate.csv
    ├── ocean_climate_oscillations.csv
    ├── environ_vars.csv
    ├── datasus_population_2001_2025.csv
    ├── shape_muni.gpkg
    ├── shape_regional_health.gpkg
    ├── shape_macroregional_health.gpkg
    └── map_regional_health.csv
"""

import pandas as pd
import geopandas as gpd
from pathlib import Path

In [2]:
# ---------------------------------------------------------------------------
# Caminhos
# ---------------------------------------------------------------------------
BASE_DIR = Path(r"C:\myenv\env\Mestrado\2026_imdc_challenge")

DB_DIR  = BASE_DIR / "0_databases"

In [3]:
# ---------------------------------------------------------------------------
# 1. dengue.csv  →  base principal
# ---------------------------------------------------------------------------
print("Carregando dengue.csv...")
dengue = pd.read_csv(
    DB_DIR / "dengue.csv",
    parse_dates=["date"],
)

# Seleciona e garante tipos das colunas de interesse
dengue = dengue[[
    "date", "epiweek", "casos", "geocode", "uf", "uf_code",
    "train_1", "train_2", "train_3", "train_4",
    "target_1", "target_2", "target_3", "target_4",
]]

# Converte tipos
dengue["date"]     = pd.to_datetime(dengue["date"]).dt.normalize()       # YYYY-MM-DD
dengue["epiweek"]  = dengue["epiweek"].astype(int)
dengue["casos"]    = dengue["casos"].astype("Int64")   # nullable int (aceita NaN)
dengue["geocode"]  = dengue["geocode"].astype(int)
dengue["uf"]       = dengue["uf"].astype(str)
dengue["uf_code"]  = dengue["uf_code"].astype(int)

# Colunas bool
bool_cols = ["train_1","train_2","train_3","train_4",
             "target_1","target_2","target_3","target_4"]
for c in bool_cols:
    dengue[c] = dengue[c].astype(bool)

# Coluna year derivada de date
dengue.insert(dengue.columns.get_loc("date") + 1, "year", dengue["date"].dt.year.astype(int))

print(f"  dengue:  {dengue.shape[0]:,} linhas  |  {dengue.shape[1]} colunas")
dengue.head()

Carregando dengue.csv...
  dengue:  4,706,650 linhas  |  15 colunas


,date,year,epiweek,casos,geocode,uf,uf_code,train_1,train_2,train_3,train_4,target_1,target_2,target_3,target_4
0,2010-01-03,2010,201001,5,3301900,RJ,33,True,True,True,True,False,False,False,False
1,2010-01-03,2010,201001,0,1504505,PA,15,True,True,True,True,False,False,False,False
2,2010-01-03,2010,201001,0,1503077,PA,15,True,True,True,True,False,False,False,False
3,2010-01-03,2010,201001,0,2106409,MA,21,True,True,True,True,False,False,False,False
4,2010-01-03,2010,201001,0,3135704,MG,31,True,True,True,True,False,False,False,False


In [4]:
dengue['date'].dt.day_name().value_counts()

date
Sunday    4706650
Name: count, dtype: int64

In [5]:
# ---------------------------------------------------------------------------
# 2.1. climate.csv  →  join por date + geocode
# ---------------------------------------------------------------------------
print("Carregando climate.csv...")
climate_cols = [
    "date", "epiweek", "geocode",
    "temp_med", "precip_med", "pressure_med", 
    "rel_humid_med", "thermal_range", "rainy_days",
]
climate = pd.read_csv(DB_DIR / "climate.csv", parse_dates=["date"], usecols=climate_cols)
climate["date"]    = pd.to_datetime(climate["date"]).dt.normalize()
climate["geocode"] = climate["geocode"].astype(int)
climate["rainy_days"] = climate["rainy_days"].astype("Int64")

# Mantém só as colunas que serão adicionadas (evita duplicar date/epiweek/geocode)
climate_add = climate[["date","geocode",
                        "temp_med","precip_med", "pressure_med",
                        "rel_humid_med","thermal_range","rainy_days"]]
print(f"  climate: {climate.shape[0]:,} linhas")
climate.head()

Carregando climate.csv...
  climate: 7,621,223 linhas


,date,epiweek,geocode,temp_med,precip_med,pressure_med,rel_humid_med,thermal_range,rainy_days
0,1999-12-26,199952,2700102,25.611700,0.0218,0.955200,67.179900,11.442500,1
1,2000-01-02,200001,2700102,25.499886,2.6425,0.955986,70.016357,8.558671,7
2,2000-01-09,200002,2700102,25.022900,9.1152,0.956700,72.211186,8.851500,7
3,2000-01-16,200003,2700102,26.944043,6.7589,0.956557,65.206800,9.307029,7
4,2000-01-23,200004,2700102,27.036471,5.7635,0.957214,66.100400,9.856343,6


In [6]:
climate['date'].dt.day_name().value_counts()

date
Sunday    7621223
Name: count, dtype: int64

In [7]:
# ---------------------------------------------------------------------------
# 2.2. ocean_climate_oscillations.csv  →  join por date
#       A base começa na Segunda; dengue começa no Domingo.
#       Ajuste: date_ocean - 1 dia  →  alinha com o Domingo da dengue.
# ---------------------------------------------------------------------------
print("Carregando ocean_climate_oscillations.csv...")
ocean = pd.read_csv(DB_DIR / "ocean_climate_oscillations.csv", parse_dates=["date"],
                    usecols=["date","enso"])
ocean["date"] = pd.to_datetime(ocean["date"]).dt.normalize()

# Domingo anterior: subtrai (dayofweek + 1) % 7 dias
# Segunda (0) → -1  |  Terça (1) → -2  | ... |  Sábado (5) → -6  |  Domingo (6) → -0
ocean["date"] = ocean["date"] - pd.to_timedelta(
    (ocean["date"].dt.dayofweek + 1) % 7, unit="D"
)
ocean["enso"] = ocean["enso"].astype(float)
print(f"  ocean:   {ocean.shape[0]:,} linhas  (date ajustado -1 dia)")

ocean.head()

Carregando ocean_climate_oscillations.csv...
  ocean:   1,721 linhas  (date ajustado -1 dia)


,date,enso
0,1993-01-03,1.169212
1,1993-01-10,1.164812
2,1993-01-17,1.041243
3,1993-01-24,1.002498
4,1993-01-31,1.112526


In [8]:
ocean['date'].dt.day_name().value_counts()

date
Sunday    1721
Name: count, dtype: int64

In [9]:
# Checagem: todos os valores devem ser 6 (domingo)
assert (ocean["date"].dt.dayofweek == 6).all(), "Ainda há datas que não são domingo!"
print("✓ Todas as datas agora apontam para o domingo anterior correspondente")

✓ Todas as datas agora apontam para o domingo anterior correspondente


In [10]:
# ---------------------------------------------------------------------------
# 2.3. environ_vars.csv  →  join por geocode  +  one-hot encode
# ---------------------------------------------------------------------------
print("Carregando environ_vars.csv...")
environ = pd.read_csv(DB_DIR / "environ_vars.csv",
                      usecols=["geocode","koppen","biome"])
environ["geocode"] = environ["geocode"].astype(int)

# One-hot: koppen  →  colunas "<valor>_koppen"  (dtype int8: 0/1)
koppen_dummies = pd.get_dummies(environ["koppen"], prefix="", prefix_sep="")\
                   .rename(columns=lambda c: f"{c}_koppen")\
                   .astype("int8")

# One-hot: biome   →  colunas "<valor>_biome"
biome_dummies  = pd.get_dummies(environ["biome"],  prefix="", prefix_sep="")\
                   .rename(columns=lambda c: f"{c}_biome")\
                   .astype("int8")

environ_ohe = pd.concat(
    [environ[["geocode"]], koppen_dummies, biome_dummies], axis=1
)
print(f"  environ: {environ_ohe.shape[0]:,} linhas  |  "
      f"{environ_ohe.shape[1]-1} colunas one-hot geradas")

environ_ohe.head()

Carregando environ_vars.csv...
  environ: 5,570 linhas  |  15 colunas one-hot geradas


,geocode,Af_koppen,Am_koppen,As_koppen,Aw_koppen,BSh_koppen,Cfa_koppen,Cfb_koppen,Cwa_koppen,Cwb_koppen,Amazônia_biome,Caatinga_biome,Cerrado_biome,Mata Atlântica_biome,Pampa_biome,Pantanal_biome
0,1100015,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
1,1100023,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
2,1100031,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
3,1100049,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
4,1100056,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0


In [11]:
# ---------------------------------------------------------------------------
# 2.4. datasus_population_2001_2025.csv  →  join por geocode + year
# ---------------------------------------------------------------------------
print("Carregando datasus_population_2001_2025.csv...")
pop = pd.read_csv(DB_DIR / "datasus_population_2001_2025.csv",
                  usecols=["geocode","year","population"])
pop["geocode"]    = pop["geocode"].astype(int)
pop["year"]       = pop["year"].astype(int)
pop["population"] = pop["population"].astype("Int64")
print(f"  population: {pop.shape[0]:,} linhas")

pop.head()

Carregando datasus_population_2001_2025.csv...
  population: 139,251 linhas


,geocode,year,population
0,1100015,2001,26553
1,3540408,2001,4496
2,3540309,2001,2567
3,3540259,2001,3637
4,3540200,2001,31595


In [12]:
# ---------------------------------------------------------------------------
# 2.5. shape_muni.gpkg  →  join por geocode
# ---------------------------------------------------------------------------
print("Carregando shape_muni.gpkg...")
muni = gpd.read_file(DB_DIR / "shape_muni.gpkg")   # lê TUDO, sem filtrar colunas aqui
 
# Guarda o CRS antes de qualquer manipulação
muni_crs = muni.crs
 
# Renomeia colunas de dados
muni = muni.rename(columns={"geocode_name": "city_name"})
muni["geocode"] = muni["geocode"].astype(int)
 
# Renomeia a coluna de geometria via rename_geometry (mantém o GeoDataFrame válido)
muni = muni.rename_geometry("city_geometry")
 
muni_add = muni[["geocode", "city_name", "city_geometry"]].copy()
print(f"  shape_muni: {muni_add.shape[0]:,} municípios  |  CRS: {muni_crs}")

muni.head()

Carregando shape_muni.gpkg...
  shape_muni: 5,570 municípios  |  CRS: EPSG:4674


,geocode,city_name,uf,uf_code,city_geometry
0,1100015,Alta Floresta D'oeste,RO,11,"MULTIPOLYGON (((-61.96836 -12.13407, -61.96827..."
1,1100023,Ariquemes,RO,11,"MULTIPOLYGON (((-63.18282 -10.13892, -63.18615..."
2,1100031,Cabixi,RO,11,"MULTIPOLYGON (((-60.70518 -13.32474, -60.70904..."
3,1100049,Cacoal,RO,11,"MULTIPOLYGON (((-61.3474 -11.50562, -61.34569 ..."
4,1100056,Cerejeiras,RO,11,"MULTIPOLYGON (((-60.82417 -13.11156, -60.82289..."


In [13]:
# ---------------------------------------------------------------------------
# 2.6. map_regional_health.csv  →  join por geocode
# ---------------------------------------------------------------------------
print("Carregando map_regional_health.csv...")
regional_map = pd.read_csv(DB_DIR / "map_regional_health.csv",
                           usecols=["geocode","macroregion_code","macroregion_name", 
                                    "regional_geocode"])
regional_map["geocode"]          = regional_map["geocode"].astype(int)
regional_map["macroregion_code"] = regional_map["macroregion_code"].astype(int)
regional_map["macroregion_name"] = regional_map["macroregion_name"].astype(str)
regional_map["regional_geocode"] = regional_map["regional_geocode"].astype(int)
print(f"  regional_map: {regional_map.shape[0]:,} linhas")

regional_map.head()

Carregando map_regional_health.csv...
  regional_map: 5,570 linhas


,macroregion_code,macroregion_name,regional_geocode,geocode
0,1,Norte,12002,1200013
1,1,Norte,12001,1200054
2,1,Norte,12001,1200104
3,1,Norte,12002,1200138
4,1,Norte,12002,1200179


In [14]:
# ---------------------------------------------------------------------------
# 2.7. shape_regional_health.gpkg  →  join por geocode
# ---------------------------------------------------------------------------
print("Carregando shape_muni.gpkg...")
region = gpd.read_file(DB_DIR / "shape_regional_health.gpkg")   # lê TUDO, sem filtrar colunas aqui
 
# Guarda o CRS antes de qualquer manipulação
region_crs = region.crs
 
# Renomeia colunas de dados
region = region.rename(columns={"regional_name": "regional_health_name"})
region["regional_geocode"] = region["regional_geocode"].astype(int)
 
# Renomeia a coluna de geometria via rename_geometry (mantém o GeoDataFrame válido)
region = region.rename_geometry("regional_health_geometry")
 
region_add = region[["regional_geocode", "regional_health_name", "regional_health_geometry"]].copy()
print(f"  shape_region: {region_add.shape[0]:,} regiões  |  CRS: {region_crs}")

region.head()

Carregando shape_muni.gpkg...
  shape_region: 439 regiões  |  CRS: EPSG:4674


,regional_geocode,regional_health_name,uf_code,regional_health_geometry
0,11001,VALE DO JAMARI,11,"MULTIPOLYGON (((-61.74735 -9.49998, -61.83013 ..."
1,11002,CAFE,11,"MULTIPOLYGON (((-61.613 -11.58384, -61.61556 -..."
2,11003,CENTRAL,11,"MULTIPOLYGON (((-61.632 -10.98629, -61.63236 -..."
3,11004,MADEIRA-MAMORE,11,"MULTIPOLYGON (((-63.75045 -9.3965, -63.74556 -..."
4,11005,ZONA DA MATA,11,"MULTIPOLYGON (((-61.94323 -11.29353, -61.92387..."


In [15]:
# ---------------------------------------------------------------------------
# 3. Montagem da forecast_database  (left joins sequenciais)
# ---------------------------------------------------------------------------
print("\nMontando forecast_database...")

forecast_database = dengue.copy()

# 2.1 climate
forecast_database = forecast_database.merge(
    climate_add, on=["date","geocode"], how="left"
)

# 2.2 ocean oscillations
forecast_database = forecast_database.merge(
    ocean, on="date", how="left"
)

# 2.3 environ vars (one-hot)
forecast_database = forecast_database.merge(
    environ_ohe, on="geocode", how="left"
)

# 2.4 population
forecast_database = forecast_database.merge(
    pop, on=["geocode","year"], how="left"
)

# 2.5 shape_muni
forecast_database = pd.merge(forecast_database, muni_add, on="geocode", how="left")
forecast_database = gpd.GeoDataFrame(
    forecast_database,
    geometry="city_geometry",
    crs=muni_crs,
)

print("Calculando área dos municípios (km²)...")

# Extrai geometrias únicas por geocode (evita recalcular para cada semana)
geom_unica = (
    forecast_database[["geocode", "city_geometry"]]
    .drop_duplicates(subset="geocode")
    .copy()
)

# Converte para GeoDataFrame temporário e reprojeta para SIRGAS 2000 / Polycônica
# EPSG:5880 → unidade em metros, projeção oficial IBGE para o Brasil
geom_proj = gpd.GeoDataFrame(geom_unica, geometry="city_geometry", crs=muni_crs)
geom_proj = geom_proj.to_crs(epsg=5880)

# Calcula área: .area retorna m² → divide por 1_000_000 para km²
geom_proj["city_area_km2"] = (geom_proj["city_geometry"].area / 1_000_000).round(4)

# Junta de volta na base principal (apenas a coluna nova)
area_map = geom_proj[["geocode", "city_area_km2"]]
forecast_database = forecast_database.merge(area_map, on="geocode", how="left")

print(f"  Área calculada para {area_map.shape[0]:,} municípios únicos")
print(f"  Exemplo — min: {area_map['city_area_km2'].min():,.1f} km²  |  "
      f"max: {area_map['city_area_km2'].max():,.1f} km²  |  "
      f"média: {area_map['city_area_km2'].mean():,.1f} km²")

# Calcula a densidade populacional
forecast_database["pop_density_km2"] = (
    forecast_database["population"] / forecast_database["city_area_km2"]
).round(4)

# Retirada da coluna city_geometry que apenas possui as coordenadas
forecast_database = forecast_database.drop(columns=["city_geometry"])

# 2.6 map_regional_health
forecast_database = forecast_database.merge(
    regional_map, on="geocode", how="left"
)

# 2.7 shape_region_health
forecast_database = pd.merge(forecast_database, region_add, on="regional_geocode", how="left")
forecast_database = gpd.GeoDataFrame(
    forecast_database,
    geometry="regional_health_geometry",
    crs=region_crs,
)

print("Calculando área das regiões (km²)...")

# Extrai geometrias únicas por geocode (evita recalcular para cada semana)
geom_unica_v2 = (
    forecast_database[["regional_geocode", "regional_health_geometry"]]
    .drop_duplicates(subset="regional_geocode")
    .copy()
)

# Converte para GeoDataFrame temporário e reprojeta para SIRGAS 2000 / Polycônica
# EPSG:5880 → unidade em metros, projeção oficial IBGE para o Brasil
geom_proj_v2 = gpd.GeoDataFrame(geom_unica_v2, geometry="regional_health_geometry", crs=region_crs)
geom_proj_v2 = geom_proj_v2.to_crs(epsg=5880)

# Calcula área: .area retorna m² → divide por 1_000_000 para km²
geom_proj_v2["regional_health_area_km2"] = (geom_proj_v2["regional_health_geometry"].area / 1_000_000).round(4)

# Junta de volta na base principal (apenas a coluna nova)
area_map_v2 = geom_proj_v2[["regional_geocode", "regional_health_area_km2"]]
forecast_database = forecast_database.merge(area_map_v2, on="regional_geocode", how="left")

print(f"  Área calculada para {area_map_v2.shape[0]:,} regiões únicas")
print(f"  Exemplo — min: {area_map_v2['regional_health_area_km2'].min():,.1f} km²  |  "
      f"max: {area_map_v2['regional_health_area_km2'].max():,.1f} km²  |  "
      f"média: {area_map_v2['regional_health_area_km2'].mean():,.1f} km²")

# Retirada da coluna regional_health_geometry que apenas possui as coordenadas
forecast_database = forecast_database.drop(columns=["regional_health_geometry"])

# ---------------------------------------------------------------------------
# 4. Resumo final
# ---------------------------------------------------------------------------
print("\n" + "="*60)
print("forecast_database  — resumo")
print("="*60)
print(f"  Linhas  : {len(forecast_database):,}")
print(f"  Colunas : {forecast_database.shape[1]}")
print(f"\n  Dtypes:\n{forecast_database.dtypes.to_string()}")
print("\n  Primeiras linhas:")
print(forecast_database.head(3).to_string())




Montando forecast_database...
Calculando área dos municípios (km²)...
  Área calculada para 5,570 municípios únicos
  Exemplo — min: 3.4 km²  |  max: 159,549.5 km²  |  média: 1,542.8 km²
Calculando área das regiões (km²)...
  Área calculada para 439 regiões únicas
  Exemplo — min: 328.7 km²  |  max: 371,921.4 km²  |  média: 19,573.9 km²

forecast_database  — resumo
  Linhas  : 4,706,650
  Colunas : 46

  Dtypes:
date                        datetime64[us]
year                                 int64
epiweek                              int64
casos                                Int64
geocode                              int64
uf                                     str
uf_code                              int64
train_1                               bool
train_2                               bool
train_3                               bool
train_4                               bool
target_1                              bool
target_2                              bool
target_3                

In [16]:
# ---------------------------------------------------------------------------
# 5. Exportação  (opcional — descomente o formato desejado)
# ---------------------------------------------------------------------------
print("Exportando o arquivo...")
import pyarrow
output_dir = DB_DIR

# --- Parquet (recomendado: preserva geometry e dtypes, menor tamanho)
forecast_database.to_parquet(output_dir / "forecast_database.parquet", index=False)

print("\nScript concluído.")

# --- CSV simples (sem geometry)
# forecast_database.drop(columns=["city_geometry"]).to_csv(
#     output_dir / "forecast_database.csv", index=False
# )

# --- GeoPackage (preserva geometry)
# forecast_database.to_file(output_dir / "forecast_database.gpkg", driver="GPKG")

# print("\nScript concluído. Descomente a seção de exportação para salvar o arquivo.")

Exportando o arquivo...

Script concluído.


### 2. Hierarchical Database

In [17]:
# ---------------------------------------------------------------------------
# Hierarquização: município → regional de saúde
# Base resultante: hierarch_forecast
# ---------------------------------------------------------------------------
fdb = forecast_database
koppen_cols = [c for c in fdb.columns if c.endswith("_koppen")]
biome_cols  = [c for c in fdb.columns if c.endswith("_biome")]
dummy_cols  = koppen_cols + biome_cols

# Chaves do groupby: regional_geocode + todas as colunas de contexto
GROUPBY_KEYS = [
    "regional_geocode",
    "date", "year", "epiweek",
    "uf", "uf_code",
    "train_1", "train_2", "train_3", "train_4",
    "target_1", "target_2", "target_3", "target_4",
    "macroregion_code", "macroregion_name",
    "regional_health_name",
]

# Agrupamento com agg dict
agg_dict = {
    "casos"      : "sum",
    "population" : "sum",
    "city_area_km2": "sum",
    "temp_med"   : "mean",
    "precip_med" : "mean",
    "pressure_med": "mean",
    "rel_humid_med": "mean",
    "thermal_range": "mean",
    "rainy_days" : "mean",
    "enso"       : "mean",
    **{c: "sum" for c in dummy_cols},   # dummies: soma → clip depois
}

hierarch_forecast = (
    fdb
    .groupby(GROUPBY_KEYS, sort=True, observed=True)
    .agg(agg_dict)
    .reset_index()
)

# Dummies: clip em 1 (soma > 1 vira 1 — presença regional)
hierarch_forecast[dummy_cols] = hierarch_forecast[dummy_cols].clip(upper=1)

# Renomeia city_area_km2 → regional_health_area_km2
hierarch_forecast = hierarch_forecast.rename(
    columns={"city_area_km2": "regional_health_area_km2"}
)

# Recria pop_density_km2
hierarch_forecast["pop_density_km2"] = (
    hierarch_forecast["population"] / hierarch_forecast["regional_health_area_km2"]
).round(4)

# Resumo
print(f"hierarch_forecast: {hierarch_forecast.shape[0]:,} linhas x {hierarch_forecast.shape[1]} colunas")
print(f"Regiões de saúde únicas: {hierarch_forecast['regional_geocode'].nunique()}")
print(f"Datas únicas           : {hierarch_forecast['date'].nunique()}")
print(hierarch_forecast.head(3).to_string())

hierarch_forecast: 370,955 linhas x 43 colunas
Regiões de saúde únicas: 439
Datas únicas           : 845
   regional_geocode       date  year  epiweek  uf  uf_code  train_1  train_2  train_3  train_4  target_1  target_2  target_3  target_4  macroregion_code macroregion_name regional_health_name  casos  population  regional_health_area_km2   temp_med  precip_med  pressure_med  rel_humid_med  thermal_range  rainy_days      enso  Af_koppen  Am_koppen  As_koppen  Aw_koppen  BSh_koppen  Cfa_koppen  Cfb_koppen  Cwa_koppen  Cwb_koppen  Amazônia_biome  Caatinga_biome  Cerrado_biome  Mata Atlântica_biome  Pampa_biome  Pantanal_biome  pop_density_km2
0             11001 2010-01-03  2010   201001  RO       11     True     True     True     True     False     False     False     False                 1            Norte       VALE DO JAMARI    219      225471                32146.8562  25.702343   26.585089      0.976689      88.954359       5.764468         7.0  0.903544          0          1     

In [18]:
# ---------------------------------------------------------------------------
# 5. Exportação  (opcional — descomente o formato desejado)
# ---------------------------------------------------------------------------
print("Exportando o arquivo...")
import pyarrow
output_dir = DB_DIR

# --- Parquet (recomendado: preserva geometry e dtypes, menor tamanho)
hierarch_forecast.to_parquet(output_dir / "hierarch_forecast.parquet", index=False)

print("\nScript concluído.")

Exportando o arquivo...

Script concluído.


In [19]:
# Verificação: total de casos deve ser igual nas duas bases
assert fdb["casos"].sum() == hierarch_forecast["casos"].sum(), "Soma de casos diverge!"
print("✓ Soma de casos consistente entre as bases.")

✓ Soma de casos consistente entre as bases.
